<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/06)_Encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 기존 파일 구조 확인

for path in sorted(SRC_DIR.glob("*.py")):
    print(path.name)

attention.py
feed_forward.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# 기존 EncoderLayer 확인

ENCODER_LAYER_PATH = SRC_DIR / "encoder_layer.py"

print(
    ENCODER_LAYER_PATH.read_text(
        encoding="utf-8"
    )
)


import torch.nn as nn

from src.multi_head_attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


class EncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.feed_forward = PositionwiseFeedForward(
            d_model,
            d_ff,
        )

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # 1. Multi-Head Self-Attention
        attention_output, attention_weights = self.self_attention(
            x,
            x,
            x,
            mask,
        )

        # 2. 첫 번째 Add & Norm
        attention_output = self.dropout1(attention_output)

      

In [ ]:
# 현재 EncoderLayer을 통과해도 입력과 출력의 shape가 동일하기 때문에 layer가 늘어나도 Tensor의 형태는 유지된다(내용만 변함).

In [ ]:
# src.encoder.py 전체 코드

import torch.nn as nn

from src.encoder_layer import EncoderLayer


class Encoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        # nn.ModuleList()를 이용해 submodule로 등록(Pytorch에서는 submodule로 등록해주지 않으면 가중치 학습이 불가하다.)
        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout,
            )
            for _ in range(num_layers) # 각 layer별로 개별적인 가중치를 만들어준다.
        ])


    def forward(self, x, mask=None):
        attention_weights_list = []

        for layer in self.layers:
            x, attention_weights = layer(
                x,
                mask,
            )

            attention_weights_list.append(
                attention_weights
            )

        return x, attention_weights_list

In [ ]:
# src/encoder.py 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/encoder.py

import torch.nn as nn

from src.encoder_layer import EncoderLayer


class Encoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout,
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        attention_weights_list = []

        for layer in self.layers:
            x, attention_weights = layer(
                x,
                mask,
            )

            attention_weights_list.append(
                attention_weights
            )

        return x, attention_weights_list

Writing /content/drive/MyDrive/attention_is_all_you_need/src/encoder.py


In [ ]:
# Encoder import

import torch

from src.encoder import Encoder

print("Encoder import 완료")

Encoder import 완료


In [ ]:
# 기본 설정

torch.manual_seed(42)

batch_size = 2
seq_len = 4

d_model = 8
num_heads = 2
d_ff = 32

num_layers = 3

dropout = 0.0

# Encoder 생성

encoder = Encoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=dropout,
)

print(encoder)

Encoder(
  (layers): ModuleList(
    (0-2): 3 x EncoderLayer(
      (self_attention): MultiHeadAttention(
        (W_Q): Linear(in_features=8, out_features=8, bias=True)
        (W_K): Linear(in_features=8, out_features=8, bias=True)
        (W_V): Linear(in_features=8, out_features=8, bias=True)
        (attention): ScaledDotProductAttention()
        (W_O): Linear(in_features=8, out_features=8, bias=True)
      )
      (feed_forward): PositionwiseFeedForward(
        (linear1): Linear(in_features=8, out_features=32, bias=True)
        (linear2): Linear(in_features=32, out_features=8, bias=True)
      )
      (dropout1): Dropout(p=0.0, inplace=False)
      (dropout2): Dropout(p=0.0, inplace=False)
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
    )
  )
)


In [ ]:
# ModuleList와 Layer 개수 확인

print(
    "layers type:",
    type(encoder.layers),
)

print(
    "number of layers:",
    len(encoder.layers),
)

layers type: <class 'torch.nn.modules.container.ModuleList'>
number of layers: 3


In [ ]:
# 각각의 EncoderLayer가 다른 객체인지 확인

print(
    encoder.layers[0]
    is encoder.layers[1]
)

# False가 나와야한다.

False


In [ ]:
# Parameter도 다른 객체인지 확인

print(
    encoder.layers[0]
    .self_attention
    .W_Q
    .weight
    is
    encoder.layers[1]
    .self_attention
    .W_Q
    .weight
)

False


In [ ]:
# 기본 Encoder Shape 테스트

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

print(
    "Encoder input shape:",
    x.shape,
)

# Encoder 실행

output, attention_weights_list = encoder(
    x,
    mask=None,
)

print(
    "Encoder output shape:",
    output.shape,
)

print(
    "Number of attention weights:",
    len(attention_weights_list),
)

Encoder input shape: torch.Size([2, 4, 8])
Encoder output shape: torch.Size([2, 4, 8])
Number of attention weights: 3


In [ ]:
# Attention Weight Shape 확인

for i, attention_weights in enumerate(
    attention_weights_list
):
    print(
        f"Layer {i + 1} attention shape:",
        attention_weights.shape,
    )

# (batch_size, num_heads, query_sequence_length, key_sequence_lemgth)

Layer 1 attention shape: torch.Size([2, 2, 4, 4])
Layer 2 attention shape: torch.Size([2, 2, 4, 4])
Layer 3 attention shape: torch.Size([2, 2, 4, 4])


In [ ]:
'''
학습 과정에서의 편의를 위해 (numlayers, batch_size, ....) 로 저장하지 않고
attention_weights_list[n] 형태로 저장한다.
'''

In [ ]:
# EncoderLayer의 shape 직접 확인

manual_x = x

print(
    "Encoder input shape:",
    manual_x.shape,
)

for i, layer in enumerate(
    encoder.layers
):
    manual_x, attention_weights = layer(
        manual_x,
        None,
    )

    print()
    print(
        f"Layer {i + 1} output shape:",
        manual_x.shape,
    )

    print(
        f"Layer {i + 1} attention shape:",
        attention_weights.shape,
    )

print()
print(
    "Final Encoder output shape:",
    manual_x.shape,
)

Encoder input shape: torch.Size([2, 4, 8])

Layer 1 output shape: torch.Size([2, 4, 8])
Layer 1 attention shape: torch.Size([2, 2, 4, 4])

Layer 2 output shape: torch.Size([2, 4, 8])
Layer 2 attention shape: torch.Size([2, 2, 4, 4])

Layer 3 output shape: torch.Size([2, 4, 8])
Layer 3 attention shape: torch.Size([2, 2, 4, 4])

Final Encoder output shape: torch.Size([2, 4, 8])


In [ ]:
# batch_size, sequence 길이 변경 테스트

# batch_size = 2, seq_len = 4에 종속되서는 안된다

test_shapes = [
    (1, 3),
    (2, 4),
    (3, 5),
    (4, 7),
]

for batch_size, seq_len in test_shapes:

    x_test = torch.randn(
        batch_size,
        seq_len,
        d_model,
    )

    output_test, attention_list_test = encoder(
        x_test,
        mask=None,
    )

    print(
        f"(B, S) = ({batch_size}, {seq_len})"
    )

    print(
        "  output:",
        output_test.shape,
    )

    for i, attention in enumerate(
        attention_list_test
    ):
        print(
            f"  layer {i + 1} attention:",
            attention.shape,
        )

    print()

(B, S) = (1, 3)
  output: torch.Size([1, 3, 8])
  layer 1 attention: torch.Size([1, 2, 3, 3])
  layer 2 attention: torch.Size([1, 2, 3, 3])
  layer 3 attention: torch.Size([1, 2, 3, 3])

(B, S) = (2, 4)
  output: torch.Size([2, 4, 8])
  layer 1 attention: torch.Size([2, 2, 4, 4])
  layer 2 attention: torch.Size([2, 2, 4, 4])
  layer 3 attention: torch.Size([2, 2, 4, 4])

(B, S) = (3, 5)
  output: torch.Size([3, 5, 8])
  layer 1 attention: torch.Size([3, 2, 5, 5])
  layer 2 attention: torch.Size([3, 2, 5, 5])
  layer 3 attention: torch.Size([3, 2, 5, 5])

(B, S) = (4, 7)
  output: torch.Size([4, 7, 8])
  layer 1 attention: torch.Size([4, 2, 7, 7])
  layer 2 attention: torch.Size([4, 2, 7, 7])
  layer 3 attention: torch.Size([4, 2, 7, 7])



In [ ]:
# num_layers 변경 테스트

for test_num_layers in [
    1,
    2,
    4,
]:

    test_encoder = Encoder(
        d_model=d_model,
        num_heads=num_heads,
        d_ff=d_ff,
        num_layers=test_num_layers,
        dropout=0.0,
    )

    x_test = torch.randn(
        2,
        4,
        d_model,
    )

    output_test, attention_list_test = (
        test_encoder(
            x_test,
            mask=None,
        )
    )

    print(
        "num_layers:",
        test_num_layers,
    )

    print(
        "output shape:",
        output_test.shape,
    )

    print(
        "attention list length:",
        len(attention_list_test),
    )

    print()

num_layers: 1
output shape: torch.Size([2, 4, 8])
attention list length: 1

num_layers: 2
output shape: torch.Size([2, 4, 8])
attention list length: 2

num_layers: 4
output shape: torch.Size([2, 4, 8])
attention list length: 4



In [ ]:
# mask = None 테스트

x_test = torch.randn(
    2,
    4,
    d_model,
)

output_test, attention_test = encoder(
    x_test,
    mask=None,
)

print(
    output_test.shape
)

torch.Size([2, 4, 8])


In [ ]:
# Encoder.forward() 수동 결과와 비교

torch.manual_seed(42)

compare_encoder = Encoder(
    d_model=8,
    num_heads=2,
    d_ff=32,
    num_layers=3,
    dropout=0.0,
)

x_compare = torch.randn(
    2,
    4,
    8,
)

# 정상적인 forwrad() 호출

encoder_output, encoder_attentions = (
    compare_encoder(
        x_compare,
        mask=None,
    )
)

# 직접 순회

manual_x = x_compare

manual_attentions = []

for layer in compare_encoder.layers:

    manual_x, attention_weights = layer(
        manual_x,
        None,
    )

    manual_attentions.append(
        attention_weights
    )

# 최종 Encdoer output 비교

print(
    "Output same:",
    torch.allclose(
        encoder_output,
        manual_x,
    )
)

# Attention도 layer 별 비교

for i in range(
    len(encoder_attentions)
):
    same = torch.allclose(
        encoder_attentions[i],
        manual_attentions[i],
    )

    print(
        f"Layer {i + 1} attention same:",
        same,
    )

In [ ]:
# 최종 통합 테스트

import torch

from src.encoder import Encoder


torch.manual_seed(42)

d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3

encoder = Encoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=0.0,
)


# ---------------------------------
# 1. ModuleList / layer 개수
# ---------------------------------

assert len(encoder.layers) == num_layers


# ---------------------------------
# 2. 서로 다른 EncoderLayer 객체
# ---------------------------------

assert (
    encoder.layers[0]
    is not
    encoder.layers[1]
)


# ---------------------------------
# 3. 서로 다른 Parameter 객체
# ---------------------------------

param0 = next(
    encoder.layers[0].parameters()
)

param1 = next(
    encoder.layers[1].parameters()
)

assert param0 is not param1


# ---------------------------------
# 4. 기본 Shape 테스트
# ---------------------------------

x = torch.randn(
    2,
    4,
    d_model,
)

output, attention_list = encoder(
    x,
    mask=None,
)

assert output.shape == (
    2,
    4,
    d_model,
)

assert len(attention_list) == num_layers

for attention in attention_list:
    assert attention.shape == (
        2,
        num_heads,
        4,
        4,
    )


# ---------------------------------
# 5. batch / sequence 변경
# ---------------------------------

test_shapes = [
    (1, 3),
    (2, 4),
    (3, 5),
    (4, 7),
]

for batch_size, seq_len in test_shapes:

    x_test = torch.randn(
        batch_size,
        seq_len,
        d_model,
    )

    output_test, attention_test = encoder(
        x_test,
        mask=None,
    )

    assert output_test.shape == (
        batch_size,
        seq_len,
        d_model,
    )

    assert (
        len(attention_test)
        ==
        num_layers
    )

    for attention in attention_test:
        assert attention.shape == (
            batch_size,
            num_heads,
            seq_len,
            seq_len,
        )


# ---------------------------------
# 6. num_layers 변경
# ---------------------------------

for test_num_layers in [
    1,
    2,
    4,
]:

    test_encoder = Encoder(
        d_model=d_model,
        num_heads=num_heads,
        d_ff=d_ff,
        num_layers=test_num_layers,
        dropout=0.0,
    )

    x_test = torch.randn(
        2,
        4,
        d_model,
    )

    output_test, attention_test = (
        test_encoder(
            x_test,
            mask=None,
        )
    )

    assert output_test.shape == (
        2,
        4,
        d_model,
    )

    assert (
        len(attention_test)
        ==
        test_num_layers
    )


# ---------------------------------
# 7. forward vs 수동 순회
# ---------------------------------

compare_encoder = Encoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=3,
    dropout=0.0,
)

x_compare = torch.randn(
    2,
    4,
    d_model,
)

encoder_output, encoder_attentions = (
    compare_encoder(
        x_compare,
        mask=None,
    )
)

manual_x = x_compare
manual_attentions = []

for layer in compare_encoder.layers:

    manual_x, attention = layer(
        manual_x,
        None,
    )

    manual_attentions.append(
        attention
    )


assert torch.allclose(
    encoder_output,
    manual_x,
)

for encoder_attention, manual_attention in zip(
    encoder_attentions,
    manual_attentions,
):
    assert torch.allclose(
        encoder_attention,
        manual_attention,
    )


print("모든 Encoder 테스트 통과")